In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from neo4j_graphrag.generation.prompts import ERExtractionTemplate

prompt_template = ERExtractionTemplate()

prompt = prompt_template.format(
    schema='',
    text = '''
        입력 텍스트
    ''',
    examples=''
)
print(prompt)


You are a top-tier algorithm designed for extracting
information in structured formats to build a knowledge graph.

Extract the entities (nodes) and specify their type from the following text.
Also extract the relationships between these nodes.

Return result as JSON using the following format:
{"nodes": [ {"id": "0", "label": "Person", "properties": {"name": "John"} }],
"relationships": [{"type": "KNOWS", "start_node_id": "0", "end_node_id": "1", "properties": {"since": "2024-08-01"} }] }

Use only the following node and relationship types (if provided):


Assign a unique ID (string) to each node, and reuse it to define relationships.
Do respect the source and target node types for relationship and
the relationship direction.

Make sure you adhere to the following rules to produce valid JSON objects:
- Do not return any additional information other than the JSON in it.
- Omit any backticks around the JSON - simply output the JSON on its own.
- The JSON object must not wrapped into a 

In [3]:
from neo4j_graphrag.llm import OpenAILLM

prompt_template = ERExtractionTemplate()
llm = OpenAILLM(model_name='gpt-4o', model_params={'temperature': 0})

input_text = '''
    빈수와 수지는 친구입니다. 빈수는 수지를 좋아합니다.
'''
prompt = prompt_template.format(
    schema = '',
    text=input_text,
    examples=''
)

response = llm.invoke(prompt)

In [4]:
print(response.content)

{
  "nodes": [
    {"id": "0", "label": "Person", "properties": {"name": "빈수"}},
    {"id": "1", "label": "Person", "properties": {"name": "수지"}}
  ],
  "relationships": [
    {"type": "FRIENDS_WITH", "start_node_id": "0", "end_node_id": "1", "properties": {}},
    {"type": "LIKES", "start_node_id": "0", "end_node_id": "1", "properties": {}}
  ]
}


In [5]:
input_text = '''
    나연은 10월 10일에 후쿠오카에 다녀왔습니다.
    후쿠오카의 주요 지하철 역인 텐진역과 하카타역 주변에서 주로 시간을 보냈습니다.
    10월 12일에 한국에 돌아왔습니다.
'''
prompt = prompt_template.format(
    schema = '',
    text = input_text,
    examples = ''
)

response = llm.invoke(prompt)
print(response.content)

{
  "nodes": [
    {
      "id": "0",
      "label": "Person",
      "properties": {
        "name": "나연"
      }
    },
    {
      "id": "1",
      "label": "Location",
      "properties": {
        "name": "후쿠오카"
      }
    },
    {
      "id": "2",
      "label": "Location",
      "properties": {
        "name": "텐진역"
      }
    },
    {
      "id": "3",
      "label": "Location",
      "properties": {
        "name": "하카타역"
      }
    },
    {
      "id": "4",
      "label": "Location",
      "properties": {
        "name": "한국"
      }
    }
  ],
  "relationships": [
    {
      "type": "VISITED",
      "start_node_id": "0",
      "end_node_id": "1",
      "properties": {
        "date": "10월 10일"
      }
    },
    {
      "type": "SPENT_TIME_AT",
      "start_node_id": "0",
      "end_node_id": "2",
      "properties": {}
    },
    {
      "type": "SPENT_TIME_AT",
      "start_node_id": "0",
      "end_node_id": "3",
      "properties": {}
    },
    {
      "type": "RETURN

In [6]:
import json
import neo4j
from neo4j_graphrag.experimental.components.kg_writer import Neo4jWriter
from neo4j_graphrag.experimental.components.types import Neo4jGraph

graph_json = ''
graph_history = ''
while(True):
    user_input = input('이야기를 들려주세요 🗣️')

    if(user_input == '완성'):
        break
    prompt = prompt_template.format(
        schema = '',
        text = user_input
            + '''(Continue extracting the graph for the following Input text.
            Ensure you retain the existing nodes and relationships from the graph history
            and add only new nodes and relationships.\n
            Graph History :)''' + graph_history + ')',
        examples = ''
    )
    print(prompt)
    response = llm.invoke(prompt)
    print(response.content)
    graph_history = response.content

    graph_json = json.loads(graph_history[graph_history.find('{'):graph_history.rfind('}')+1])


You are a top-tier algorithm designed for extracting
information in structured formats to build a knowledge graph.

Extract the entities (nodes) and specify their type from the following text.
Also extract the relationships between these nodes.

Return result as JSON using the following format:
{"nodes": [ {"id": "0", "label": "Person", "properties": {"name": "John"} }],
"relationships": [{"type": "KNOWS", "start_node_id": "0", "end_node_id": "1", "properties": {"since": "2024-08-01"} }] }

Use only the following node and relationship types (if provided):


Assign a unique ID (string) to each node, and reuse it to define relationships.
Do respect the source and target node types for relationship and
the relationship direction.

Make sure you adhere to the following rules to produce valid JSON objects:
- Do not return any additional information other than the JSON in it.
- Omit any backticks around the JSON - simply output the JSON on its own.
- The JSON object must not wrapped into a 

In [7]:
print(graph_history)

{
  "nodes": [
    {"id": "0", "label": "Person", "properties": {"name": "나연"}},
    {"id": "1", "label": "Person", "properties": {"name": "도준"}},
    {"id": "2", "label": "Place", "properties": {"name": "판교역 근처의 고층 빌딩"}},
    {"id": "3", "label": "Person", "properties": {"name": "지수"}},
    {"id": "4", "label": "Person", "properties": {"name": "민혁"}}
  ],
  "relationships": [
    {"type": "WORKS_AT", "start_node_id": "0", "end_node_id": "2", "properties": {}},
    {"type": "MARRIED_TO", "start_node_id": "0", "end_node_id": "1", "properties": {}},
    {"type": "KNOWS", "start_node_id": "0", "end_node_id": "3", "properties": {}},
    {"type": "TOLD", "start_node_id": "3", "end_node_id": "0", "properties": {"about": "도준 바람"}},
    {"type": "HELPED", "start_node_id": "4", "end_node_id": "0", "properties": {}},
    {"type": "COLLEAGUE_OF", "start_node_id": "4", "end_node_id": "1", "properties": {}}
  ]
}


In [9]:
graph_json = ''
graph_history = ''

while(True):
    user_input = input('이야기를 들려주세요 🗣️')

    if(user_input == '완성'):
        with neo4j.GraphDatabase.driver('bolt://54.86.94.46:7687',auth=('neo4j', 'reveille-name-burn')) as driver:
            writer = Neo4jWriter(driver)
            graph = Neo4jGraph(
                nodes = graph_json['nodes'],
                relationships = graph_json['relationships']
            )
            await writer.run(graph)
            break
    prompt = prompt_template.format(
        schema = '',
        text = user_input
            + '''(Continue extracting the graph for the following Input text.
            Ensure you retain the existing nodes and relationships from the graph history
            and add only new nodes and relationships.\n
            Graph History :)''' + graph_history + ')',
        examples = ''
    )
    print(prompt)
    response = llm.invoke(prompt)
    print(response.content)
    graph_history = response.content

    graph_json = json.loads(graph_history[graph_history.find('{'):graph_history.rfind('}')+1])


You are a top-tier algorithm designed for extracting
information in structured formats to build a knowledge graph.

Extract the entities (nodes) and specify their type from the following text.
Also extract the relationships between these nodes.

Return result as JSON using the following format:
{"nodes": [ {"id": "0", "label": "Person", "properties": {"name": "John"} }],
"relationships": [{"type": "KNOWS", "start_node_id": "0", "end_node_id": "1", "properties": {"since": "2024-08-01"} }] }

Use only the following node and relationship types (if provided):


Assign a unique ID (string) to each node, and reuse it to define relationships.
Do respect the source and target node types for relationship and
the relationship direction.

Make sure you adhere to the following rules to produce valid JSON objects:
- Do not return any additional information other than the JSON in it.
- Omit any backticks around the JSON - simply output the JSON on its own.
- The JSON object must not wrapped into a 